# Business Entity Resolution — full training + test prediction on Colab (T4)

Runs the `okaya` pipeline end to end: download data → normalize → learn region/transliteration tables → blocking (GPU) → pruner → two-stage LightGBM → calibration/decision tuning → test prediction → validation → zip of everything you need to submit.

**Runtime → Change runtime type → T4 GPU** before running. Every stage is checkpointed under `WORK`, so if the session disconnects, just run all cells again: finished stages are skipped.

Tip: mount Google Drive and set `WORK` to a Drive folder if you want checkpoints to survive a runtime reset (slower I/O).

In [ ]:
#@title 1. Setup: clone repo, install dependencies
import os, subprocess, sys
REPO = 'https://github.com/BhavyeNijhawan/okaya.git'
if not os.path.exists('/content/okaya'):
    !git clone -q {REPO} /content/okaya
else:
    !cd /content/okaya && git pull -q
!pip install -q rapidfuzz anyascii sparse_dot_topn lightgbm pyarrow scikit-learn
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU: retrieval falls back to CPU (slower)'
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
os.chdir('/content/okaya'); sys.path.insert(0, '/content/okaya')
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)

In [ ]:
#@title 2. Data: download the official student_resource zip and extract the dataset
DATA_URL = 'https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip'  #@param {type:'string'}
DATA = '/content/data'    # will contain train/ and test/
WORK = '/content/work'    # caches, models, outputs (set to a Drive path to persist across sessions)
os.makedirs(DATA, exist_ok=True); os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f'{DATA}/train/train_source1.tsv'):
    !wget -q --show-progress -O /content/student_resource.zip {DATA_URL}
    !unzip -q -o /content/student_resource.zip -d /content/sr
    !mv /content/sr/student_resource/dataset/train {DATA}/train && mv /content/sr/student_resource/dataset/test {DATA}/test
    !rm -rf /content/sr /content/student_resource.zip
!ls -la {DATA}/train {DATA}/test
!df -h /content | tail -1

In [ ]:
#@title 3. Train (all stages are cached; safe to re-run after a disconnect)
JOBS = 2  #@param {type:'integer'}
!python run.py train --data {DATA} --work {WORK} --jobs {JOBS} 2>&1 | tee -a {WORK}/train_log.txt

In [ ]:
#@title 4. Predict on the test set and validate the submission files
OUT = '/content/output'
!python run.py predict --data {DATA} --work {WORK} --out {OUT} --jobs {JOBS} 2>&1 | tee -a {WORK}/predict_log.txt
!python utils/validate_submission.py --matching {OUT}/matching_results.tsv --candidate {OUT}/candidate_pairs.tsv --test-dir {DATA}/test
import json; print(json.dumps(json.load(open(f'{OUT}/diagnostics.json')), indent=1))
print(json.dumps(json.load(open(f'{WORK}/train/tune_report.json'))['variants'], indent=1))

In [ ]:
#@title 5. Package everything into one zip (submission layout + models + reports) and download
import shutil, glob
PKG = '/content/okaya_submission'
shutil.rmtree(PKG, ignore_errors=True)
os.makedirs(f'{PKG}/output', exist_ok=True); os.makedirs(f'{PKG}/code/business_entity_resolution', exist_ok=True); os.makedirs(f'{PKG}/models', exist_ok=True)
for f in ['matching_results.tsv', 'candidate_pairs.tsv', 'diagnostics.json']:
    shutil.copy(f'{OUT}/{f}', f'{PKG}/output/{f}')
shutil.copytree('/content/okaya/ber', f'{PKG}/code/business_entity_resolution/src/ber')
for f in ['run.py', 'requirements.txt', 'README.md']:
    shutil.copy(f'/content/okaya/{f}', f'{PKG}/code/business_entity_resolution/' + ('src/' if f == 'run.py' else '') + f)
shutil.copytree('/content/okaya/utils', f'{PKG}/code/business_entity_resolution/utils')
shutil.copy('/content/okaya/Documentation_template.md', f'{PKG}/Documentation_template.md')
for f in glob.glob(f'{WORK}/train/*.txt') + glob.glob(f'{WORK}/train/*.json') + glob.glob(f'{WORK}/tables/*.json'):
    shutil.copy(f, f'{PKG}/models/' + os.path.basename(f))
shutil.make_archive('/content/okaya_submission', 'zip', PKG)
!ls -la /content/okaya_submission.zip
from google.colab import files
files.download('/content/okaya_submission.zip')

### Notes
* `output/matching_results.tsv` is the leaderboard file; `output/candidate_pairs.tsv` is the exact candidate set the final model scored.
* Validation score of the trained model (faithful, test-density universe) is printed by the `tune` stage and stored in `WORK/train/tune_report.json`.
* To resume after a disconnect: run cells 1–2 again (fast), then 3/4 — cached stages are skipped.